In [37]:
import sys
sys.path.append('..')  # Add the parent directory to Python's search path
import pyomo.environ as pyo
import numpy as np
from pyomolayer_test import PyomoOptLayer
import torch
import time
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

In [51]:
import numpy as np
from scipy.sparse import coo_matrix
from mumps import DMumpsContext
from scipy.sparse.linalg import spsolve
# Step 1: Define sparse matrix A
# Example: 5x5 sparse matrix
row = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 0.0, 1.0])
col = np.array([0, 1, 2, 3, 4.0, 1, 2.0])
data = np.array([10, 20, 30.0, 40.0, 50, 5, 7])

# Create a sparse matrix in COO format
A = coo_matrix((data, (row, col)), shape=(5, 5))
print(A.todense())
# Step 2: Define the dense matrix B (multiple right-hand sides)
# Example: 5x3 matrix
B = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
    [10, 11, 12],
    [13, 14, 15]
], dtype=float)


[[10.  5.  0.  0.  0.]
 [ 0. 20.  7.  0.  0.]
 [ 0.  0. 30.  0.  0.]
 [ 0.  0.  0. 40.  0.]
 [ 0.  0.  0.  0. 50.]]


In [53]:
# Step 3: Solve AX = B using MUMPS
ctx = DMumpsContext()  # Initialize MUMPS context
res = np.zeros(shape = ())
for i in range(3):
    if ctx.myid == 0:  # Ensure only the root process executes this block
        ctx.set_centralized_sparse(A)  # Set the sparse matrix A
        X = B[:, i].copy()  # Create a copy of B to hold the solution X
        ctx.set_rhs(X)  # Set B as the right-hand side matrix
    ctx.run(job=6)  # Analysis, Factorization, and Solve
    ctx.destroy()  # Cleanup resources

# Step 4: Display the solution matrix X
if ctx.myid == 0:  # Only the root process displays results
    print("Solution matrix X (solves AX = B):")
    print(X)


Entering DMUMPS 5.7.3 from C interface with JOB, N, NNZ =   6           5              7
      executing #MPI =      1 and #OMP =     12

  MUMPS compiled with option -DAVOID_MPI_IN_PLACE
 MUMPS compiled with option -Dmetis
 MUMPS compiled with option -Dpord
 MUMPS compiled with option -Dscotch
L U Solver for unsymmetric matrices
Type of parallelism: Working host

 ****** ANALYSIS STEP ********

 Processing a graph of size:         5
 ... Structural symmetry (in percent)=   71
 Average density of rows/columns =    1
 ... No column permutation
 Ordering based on AMF 

Leaving analysis phase with  ...
 INFOG(1)                                       =               0
 INFOG(2)                                       =               0
 -- (20) Number of entries in factors (estim.)  =              11
 --  (3) Real space for factors    (estimated)  =              11
 --  (4) Integer space for factors (estimated)  =              70
 --  (5) Maximum frontal size      (estimated)  =             

In [54]:
# Create a sparse matrix in COO format
A = coo_matrix((data, (row, col)), shape=(5, 5))
print(A.todense())
# Step 2: Define the dense matrix B (multiple right-hand sides)
# Example: 5x3 matrix
B = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
    [10, 11, 12],
    [13, 14, 15]
], dtype=float)

spsolve(A, B[:, 0])

[[10.  5.  0.  0.  0.]
 [ 0. 20.  7.  0.  0.]
 [ 0.  0. 30.  0.  0.]
 [ 0.  0.  0. 40.  0.]
 [ 0.  0.  0.  0. 50.]]


array([0.04083333, 0.11833333, 0.23333333, 0.25      , 0.26      ])

# QP
\begin{align}
\text{min}_\mathbf{x} & \frac{1}{2} \mathbf{x}^\top \mathbf{Q} \mathbf{x} + \mathbf{q}^\top\mathbf{x}, 
\\
\\
\text{s.t.} \quad & \mathbf{A} \mathbf{x} = \mathbf{b} \\
& \mathbf{G}\mathbf{x} \leq \mathbf{h}
\end{align}
with the variable $\mathbf{x}$ and parameter $\mathbf{p}$.

In [38]:
def create_model(nominal_Psqrt, nominal_q, nominal_A, nominal_b, nominal_G, nominal_h):
    # Create a concrete model
    m = pyo.ConcreteModel()

    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    m.ipopt_zL_out = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    m.ipopt_zU_out = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    # Define variables
    m.x = pyo.Var(range(n), within=pyo.Reals)
    
    # Define parameters
    m.Psqrt = pyo.Var(range(n), range(n), within=pyo.Reals)
    m.q = pyo.Var(range(n), within=pyo.Reals)            # Linear term vector
    m.A = pyo.Var(range(t), range(n), within=pyo.Reals)  # Equality constraint matrix
    m.b = pyo.Var(range(t), within=pyo.Reals)            # Equality constraint vector
    m.G = pyo.Var(range(p), range(n), within=pyo.Reals)  # Inequality constraint matrix
    m.h = pyo.Var(range(p), within=pyo.Reals) 
    
    for i in range(n):
        for j in range(n):
            m.Psqrt[i, j].fix(nominal_Psqrt[i, j])

    for i in range(n):
        m.q[i].fix(nominal_q[i])
    
    for i in range(t):
        for j in range(n):
            m.A[i, j].fix(nominal_A[i, j])
            
    for i in range(t):  
        m.b[i].fix(nominal_b[i])

    for i in range(p):
        for j in range(n):
            m.G[i, j].fix(nominal_G[i, j])

    for i in range(p):
        m.h[i].fix(nominal_h[i])

    
    # Define equality constraints
    m.equ_constraints = pyo.ConstraintList()
    for i in range(t):
        m.equ_constraints.add(sum(m.A[i, j] * m.x[j] for j in range(n)) == m.b[i])

    # Define inequality constraints
    m.inequ_constraints = pyo.ConstraintList()
    for i in range(p):
        m.inequ_constraints.add(sum(m.G[i, j] * m.x[j] for j in range(n)) - m.h[i] <= 0)
    
    # Define objective function: 
    def objective_rule(m):
        tol_term = 0
        for i in range(n):
            q_term = 0
            for j in range(n):
                q_term += m.Psqrt[i, j] * m.x[j]
            tol_term += q_term**2
            
        return 0.5 * tol_term + sum(m.q[i] * m.x[i] for i in range(n))
        
    m.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)
     
    return m

n,t,p = 10, 5, 2

nominal_Psqrt = np.random.rand(n, n)
nominal_q = np.random.rand(n)

nominal_A = np.random.rand(t, n)
nominal_b = np.random.rand(t)
nominal_G = np.random.rand(p, n)
nominal_h = np.random.rand(p)

model = create_model(nominal_Psqrt, nominal_q, nominal_A, nominal_b, nominal_G, nominal_h)
opt = pyo.SolverFactory('ipopt', tee=True)
results = opt.solve(model) 
print(results)
#model.pprint()
print(model.obj())


Problem: 
- Lower bound: -inf
  Upper bound: inf
  Number of objectives: 1
  Number of constraints: 7
  Number of variables: 10
  Sense: unknown
Solver: 
- Status: ok
  Message: Ipopt 3.13.2\x3a Optimal Solution Found
  Termination condition: optimal
  Id: 0
  Error rc: 0
  Time: 0.051204681396484375
Solution: 
- number of solutions: 0
  number of solutions displayed: 0

1.2515546830428206


In [39]:
variables_name = ["x"]

variables_size = {'x':[n]}

parameters_name = ["Psqrt", "q", "A", "b", "G", "h"]

parameters_size = {'Psqrt':[n, n], "q" : [n], "A" : [t, n], "b": [t], 'G':[p, n], "h": [p]}

Pyomolayer = PyomoOptLayer(create_model, variables_name, variables_size, parameters_name, parameters_size, solver = 'ipopt')

In [41]:
torch.manual_seed(0)
batch_size = 10

Psqrt = torch.randn(batch_size, n, n, requires_grad=True, dtype=torch.float64)
qval = torch.randn(batch_size, n, requires_grad=True, dtype=torch.float64)

Aval = torch.randn(batch_size, t, n, requires_grad=True, dtype=torch.float64)
bval = torch.randn(batch_size, t, requires_grad=True, dtype=torch.float64)

Gval = torch.randn(batch_size, p, n, requires_grad=True, dtype=torch.float64)
hval = torch.randn(batch_size, p, requires_grad=True, dtype=torch.float64)

print(Psqrt.shape)

start = time.time()

input = tuple([Psqrt, qval, Aval, bval, Gval, hval])
primal, _, _, _, _ = Pyomolayer(*input)
primal.sum().backward()

end = time.time()

pyomo_time = end - start

torch.Size([10, 10, 10])


In [42]:
x = cp.Variable(n)

Q_sqrt = cp.Parameter((n, n))
q = cp.Parameter(n)
A = cp.Parameter((t, n))
b = cp.Parameter(t)
G = cp.Parameter((p, n))
h = cp.Parameter(p)

obj = cp.Minimize(0.5*cp.sum_squares(Q_sqrt@x) + q @ x)
cons = [A @ x == b, G @ x <= h]
prob = cp.Problem(obj, cons)
Layer = CvxpyLayer(prob, parameters=[Q_sqrt, q, A, b, G, h], variables=[x])

torch.manual_seed(0)

Psqrt_cvx = Psqrt.clone().detach().requires_grad_(True)
qval_cvx = qval.clone().detach().requires_grad_(True)

Aval_cvx = Aval.clone().detach().requires_grad_(True)
bval_cvx = bval.clone().detach().requires_grad_(True)

Gval_cvx = Gval.clone().detach().requires_grad_(True)
hval_cvx = hval.clone().detach().requires_grad_(True)


In [43]:
start = time.time()

primal_cvx, = Layer(Psqrt_cvx, qval_cvx, Aval_cvx, bval_cvx, Gval_cvx, hval_cvx)

primal_cvx.sum().backward()

end = time.time()

cvxpy_time = end - start

In [46]:
print("Primal", np.max(torch.abs(primal_cvx - primal).detach().numpy()))
print("Grad_Psqrt", np.max(torch.abs(Psqrt_cvx.grad - Psqrt.grad).detach().numpy()))
print("Grad_q", np.max(torch.abs(qval_cvx.grad - qval.grad).detach().numpy()))
print("Grad_A", np.max(torch.abs(Aval_cvx.grad - Aval.grad).detach().numpy()))
print("Grad_G", np.max(torch.abs(Gval_cvx.grad - Gval.grad).detach().numpy()))
print("Grad_h", np.max(torch.abs(hval_cvx.grad - hval.grad).detach().numpy()))

Primal 0.00029035613125971693
Grad_Psqrt 0.00024168329415894668
Grad_q 7.790894050319075e-05
Grad_A 0.0010170294741616015
Grad_G 0.000276944949154978
Grad_h 0.00012825793323101298


In [50]:
Psqrt_cvx.grad.shape

torch.Size([10, 10, 10])

In [52]:
np.sum(torch.abs(Psqrt_cvx.grad - Psqrt.grad).detach().numpy())

0.011350021108525907

In [45]:
print("test samples", batch_size)
print("var dim, equ num, inequ num", n, t, p )
print("Pyomo time (second)", pyomo_time)
print("Cvxpy time (second)", cvxpy_time)

test samples 10
var dim, equ num, inequ num 10 5 2
Pyomo time (second) 0.6539607048034668
Cvxpy time (second) 0.10212850570678711


# QCQP
\begin{align}
\text{min}_\mathbf{x} & \frac{1}{2} \mathbf{x}^\top \mathbf{Q} \mathbf{x} + \mathbf{q}^\top\mathbf{x}, 
\\
\\
\text{s.t.} \quad & \mathbf{F} \mathbf{x} = \mathbf{g} \\
& \frac{1}{2} \mathbf{x}^\top \mathbf{A} \mathbf{x} + \mathbf{b}^\top\mathbf{x} + \mathbf{d} \leq \mathbf{s}_u \\
& \mathbf{x}_l \leq \mathbf{x} \leq \mathbf{x}_u
\end{align}
with the variable $\mathbf{x}$ and parameter $\mathbf{p}$.

In [8]:
def create_model(nominal_Psqrt, nominal_q, nominal_A, nominal_b, nominal_d, nominal_F, nominal_g):
    # Create the model
    model = pyo.ConcreteModel()
    model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    model.ipopt_zL_out = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    model.ipopt_zU_out = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    # Define the decision variables
    model.x = pyo.Var(range(n), domain=pyo.Reals, bounds = (-0.1, 1))

    # Define parameters
    model.Psqrt = pyo.Var(range(n), range(n), within=pyo.Reals)
    model.q = pyo.Var(range(n), within=pyo.Reals) # Linear term vector
    model.A = pyo.Var(range(m), range(n), range(n), within=pyo.Reals)
    model.b = pyo.Var(range(m), range(n), within=pyo.Reals)
    model.d = pyo.Var(range(m), within=pyo.Reals)
    model.F = pyo.Var(range(p), range(n), within=pyo.Reals)
    model.g = pyo.Var(range(p), within=pyo.Reals)

    for i in range(n):
        for j in range(n):
            model.Psqrt[i, j].fix(nominal_Psqrt[i, j])

    for i in range(n):
        model.q[i].fix(nominal_q[i])

    for i in range(m):
        for j in range(n):
            for k in range(n):
                model.A[i, j, k].fix(nominal_A[i, j, k])

    for i in range(m):
        for j in range(n):
            model.b[i, j].fix(nominal_b[i, j])

    for i in range(m):
        model.d[i].fix(nominal_d[i])

    for i in range(p):
        for j in range(n):
            model.F[i, j].fix(nominal_F[i, j])

    for i in range(p):
        model.g[i].fix(nominal_g[i])
        
    # Define objective function: 
    def objective_rule(model):
        tol_term = 0
        for i in range(n):
            q_term = 0
            for j in range(n):
                q_term += model.Psqrt[i, j] * model.x[j]
            tol_term += q_term**2
            
        return 0.5 * tol_term + sum(model.q[i] * model.x[i] for i in range(n))

    model.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

    # Linear equality constraints: F @ x == g
    model.equ_constraints = pyo.ConstraintList()
    for i in range(p):
        model.equ_constraints.add(sum(model.F[i, j] * model.x[j] for j in range(n)) == model.g[i])

    # QC constraints
    def quadratic_constraint_rule(model, i):
        tol_term = 0
        for j in range(n):
            q_term = 0
            for k in range(n):
                q_term += model.A[i, j, k] * model.x[k]
            tol_term += q_term**2

        return (None, 0.5 * tol_term + sum(model.b[i, j] * model.x[j] for j in range(n)) + model.d[i], 1)
        
    model.inequ_constraints = pyo.ConstraintList()
    for i in range(m):
        model.inequ_constraints.add(quadratic_constraint_rule(model, i))
        
    return model
    
# np.random.seed(0)
m = 1
n = 5
p = 1

# Generate random parameters
nominal_Psqrt = np.random.rand(n, n)
nominal_q = np.random.rand(n)

nominal_A = np.random.randn(m, n, n)
nominal_b = np.random.randn(m, n)
nominal_d = np.random.randn(m)

nominal_F = np.random.randn(p, n)
nominal_g = np.random.randn(p)

model = create_model(nominal_Psqrt, nominal_q, nominal_A, nominal_b, nominal_d, nominal_F, nominal_g)
solver = pyo.SolverFactory('ipopt')

# # Solve the model
results = solver.solve(model, tee=True)

# # Display the results
model.pprint
model.display()
#model.obj()


Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

In [9]:
variables_name = ["x"]

variables_size = {'x':[n]}

parameters_name = ["Psqrt", "q", "A", "b", "d", "F", "g"]

parameters_size = {'Psqrt':[n, n], 'q':[n], "A" : [m, n, n], "b" : [m, n], 'd':[m], "F": [p, n], "g": [p]}

Pyomolayer = PyomoOptLayer(create_model, variables_name, variables_size, parameters_name, parameters_size, solver = 'ipopt')

In [10]:
torch.manual_seed(1)
batch_size = 2
Psqrtval = torch.randn(batch_size, n, n, requires_grad=True, dtype=torch.float64)
qval = torch.randn(batch_size, n, requires_grad=True, dtype=torch.float64)
Aval = torch.randn(batch_size, m, n, n, requires_grad=True, dtype=torch.float64)
bval = torch.randn(batch_size, m, n, requires_grad=True, dtype=torch.float64)
dval = torch.randn(batch_size, m, requires_grad=True, dtype=torch.float64)
Fval = torch.randn(batch_size, p, n, requires_grad=True, dtype=torch.float64)
gval= torch.randn(batch_size, p, requires_grad=True, dtype=torch.float64)


In [11]:
start = time.time()

input = tuple([Psqrtval, qval, Aval, bval, dval, Fval, gval])
primal, _, _, _ = Pyomolayer(*input)
primal.sum().backward()

end = time.time()

pyomo_time = end - start

print("pyomo_time", pyomo_time)

pyomo_time 0.13179969787597656


/kfs2/projects/drl4dsr/kchen2/PyomoLayer/tests/../utilities.py:81: RuntimeWarning: divide by zero encountered in divide
  data = duals_primals_lb / (primals - self._nlp.primals_lb()) + duals_primals_ub / (self._nlp.primals_ub() - primals)


In [12]:
x = cp.Variable(n)

Q_sqrt = cp.Parameter((n, n))
q = cp.Parameter(n)
A = cp.Parameter((n, n))
b = cp.Parameter(n)
d = cp.Parameter(1)
F = cp.Parameter((p, n))
g = cp.Parameter(p)

obj = cp.Minimize(0.5*cp.sum_squares(Q_sqrt @ x) + q @ x)
cons = [F @ x == g, 0.5*cp.sum_squares(A @ x) + b @ x + d - 1 <= 0, x - 1 <= 0, -0.1 - x <= 0]
prob = cp.Problem(obj, cons)
Layer = CvxpyLayer(prob, parameters=[Q_sqrt, q, A, b, d, F, g], variables=[x])

Psqrt_cvx = Psqrtval.clone().detach().requires_grad_(True)
qval_cvx = qval.clone().detach().requires_grad_(True)

Aval_cvx = Aval[:, 0].clone().detach().requires_grad_(True)
bval_cvx = bval[:, 0].clone().detach().requires_grad_(True)
dval_cvx = dval.clone().detach().requires_grad_(True)

Fval_cvx = Fval.clone().detach().requires_grad_(True)
gval_cvx = gval.clone().detach().requires_grad_(True)

In [13]:
start = time.time()

primal_cvx, = Layer(Psqrt_cvx, qval_cvx, Aval_cvx, bval_cvx, dval_cvx, Fval_cvx, gval_cvx)

primal_cvx.sum().backward()

end = time.time()

cvxpy_time = end - start

print("cvxpy_time", cvxpy_time)

cvxpy_time 0.1756424903869629


In [14]:
#x
print("Primal", np.max(torch.abs(primal_cvx - primal).detach().numpy()))

Primal 0.00018182636435670751


In [15]:
#Gradient
#x/Psqrt
print("Grad_Psqrt", np.max(torch.abs(Psqrt_cvx.grad - Psqrtval.grad).detach().numpy()))
#x/q
print("Grad_q", np.max(torch.abs(qval_cvx.grad - qval.grad).detach().numpy()))

print("Grad_A", np.max(torch.abs(Aval_cvx.grad - Aval.grad[:, 0]).detach().numpy()))
print("Grad_b", np.max(torch.abs(bval_cvx.grad - bval.grad[:, 0]).detach().numpy()))
print("Grad_d", np.max(torch.abs(dval_cvx.grad - dval.grad).detach().numpy()))
print("Grad_G", np.max(torch.abs(Fval_cvx.grad - Fval.grad).detach().numpy()))
print("Grad_h", np.max(torch.abs(gval_cvx.grad - gval.grad).detach().numpy()))

Grad_Psqrt 0.0004050652471547789
Grad_q 0.0002846104380970216
Grad_A 2.2943214752564395e-05
Grad_b 1.0191789768262377e-05
Grad_d 1.407839765058989e-05
Grad_G 0.0001130249301755093
Grad_h 4.558848480645805e-05
